In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, Input, Model

# -------------------------
# 1. Load CSV
# -------------------------
df = pd.read_csv("gas_data/datasets/Gas_Sensors_Measurements.csv")
sensor_cols = ['MQ2','MQ3','MQ5','MQ6','MQ7','MQ8','MQ135']

# Label mapping
labels = sorted(df['Gas'].unique())
label_mapping = {l:i for i,l in enumerate(labels)}
df['label'] = df['Gas'].map(label_mapping)

# -------------------------
# 2. Load images & sensors
# -------------------------
IMG_DIR = "gas_data/datasets/Images"
IMG_SIZE = (128,128)

X_img, X_sensor, y_labels = [], [], []

for idx, row in df.iterrows():
    img_path = os.path.join(IMG_DIR, row['Corresponding Image Name'] + ".png")
    if not os.path.exists(img_path):
        continue
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img = cv2.resize(img, IMG_SIZE)/255.0
    X_img.append(img.reshape(128,128,1))
    X_sensor.append(row[sensor_cols].values)
    y_labels.append(row['label'])

X_img = np.array(X_img)
X_sensor = np.array(X_sensor)
y = to_categorical(np.array(y_labels), num_classes=len(label_mapping))

# Scale sensor data
scaler = MinMaxScaler()
X_sensor = scaler.fit_transform(X_sensor)

# -------------------------
# 3. Build robust CNN + Sensor branch
# -------------------------
# Image branch
img_input = Input(shape=(128,128,1))
x = layers.Conv2D(32,(3,3),activation='relu',padding='same')(img_input)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(64,(3,3),activation='relu',padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(128,(3,3),activation='relu',padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Flatten()(x)
x = layers.Dense(128,activation='relu')(x)
x = layers.Dropout(0.4)(x)

# Sensor branch
sensor_input = Input(shape=(X_sensor.shape[1],))
s = layers.Dense(64,activation='relu')(sensor_input)
s = layers.BatchNormalization()(s)
s = layers.Dense(32,activation='relu')(s)

# Merge
combined = layers.concatenate([x,s])
z = layers.Dense(64,activation='relu')(combined)
z = layers.Dropout(0.3)(z)
output = layers.Dense(len(label_mapping),activation='softmax')(z)

model = Model(inputs=[img_input,sensor_input],outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# -------------------------
# 4. Train
# -------------------------
history = model.fit([X_img,X_sensor], y, epochs=5, batch_size=32, validation_split=0.2)

# -------------------------
# 5. Predict ANY new sample
# -------------------------
def predict_new_sample(image_path, sensor_vals):
    """
    Predict gas type, confidence, and threat from any new image + sensor reading
    """
    # Image preprocessing
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    img = cv2.resize(img, IMG_SIZE)/255.0
    img = img.reshape(1,128,128,1)

    # Sensor preprocessing
    sensor_arr = np.array(sensor_vals).reshape(1,-1)
    sensor_arr = scaler.transform(sensor_arr)

    # Prediction
    probs = model.predict([img,sensor_arr])[0]
    idx = np.argmax(probs)
    label = labels[idx]
    confidence = probs[idx]*100
    threat = "Dangerous" if label!="NoGas" else "Safe"

    print(f"Predicted Gas: {label}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Threat Level: {threat}")
    return label, confidence, threat

# -------------------------
# Example usage for a new sample
# -------------------------
sample_sensor = [560,518,377,339,667,452,418]  # new sensor values
sample_image = "gas_data/datasets/Images/999_Mixture.png"  # new image
predict_new_sample(sample_image, sample_sensor)


In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, Input, Model

df = pd.read_csv("gas_data/datasets/Gas_Sensors_Measurements.csv")
sensor_cols = ['MQ2','MQ3','MQ5','MQ6','MQ7','MQ8','MQ135']

labels = sorted(df['Gas'].unique())
label_mapping = {l:i for i,l in enumerate(labels)}
df['label'] = df['Gas'].map(label_mapping)

IMG_DIR = "gas_data/datasets/Images"
IMG_SIZE = (128,128)

X_img, X_sensor, y_labels = [], [], []

for idx, row in df.iterrows():
    img_path = os.path.join(IMG_DIR, row['Corresponding Image Name'] + ".png")
    if not os.path.exists(img_path):
        continue
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img = cv2.resize(img, IMG_SIZE)/255.0
    X_img.append(img.reshape(128,128,1))
    X_sensor.append(row[sensor_cols].values)
    y_labels.append(row['label'])

X_img = np.array(X_img)
X_sensor = np.array(X_sensor)
y = to_categorical(np.array(y_labels), num_classes=len(label_mapping))

scaler = MinMaxScaler()
X_sensor = scaler.fit_transform(X_sensor)

X_img_train, X_img_test, X_sensor_train, X_sensor_test, y_train, y_test = train_test_split(
    X_img, X_sensor, y, test_size=0.2, random_state=42
)

img_input = Input(shape=(128,128,1))
x = layers.Conv2D(32,(3,3),activation='relu',padding='same')(img_input)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(64,(3,3),activation='relu',padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(128,(3,3),activation='relu',padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Flatten()(x)
x = layers.Dense(128,activation='relu')(x)
x = layers.Dropout(0.4)(x)

sensor_input = Input(shape=(X_sensor.shape[1],))
s = layers.Dense(64,activation='relu')(sensor_input)
s = layers.BatchNormalization()(s)
s = layers.Dense(32,activation='relu')(s)

combined = layers.concatenate([x,s])
z = layers.Dense(64,activation='relu')(combined)
z = layers.Dropout(0.3)(z)
output = layers.Dense(len(label_mapping),activation='softmax')(z)

model = Model(inputs=[img_input,sensor_input],outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit([X_img_train,X_sensor_train], y_train,
                    epochs=10, batch_size=32, validation_split=0.2)

test_loss, test_acc = model.evaluate([X_img_test, X_sensor_test], y_test)
print(f"\n Test Accuracy: {test_acc:.2f}")

y_pred_probs = model.predict([X_img_test, X_sensor_test])
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()




Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32768)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │        512 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │  4,194,432 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 160)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │     10,304 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 4)         │        260 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 4,300,772 (16.41 MB)

 Trainable params: 4,300,516 (16.41 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 96s 3s/step - accuracy: 0.9707 - loss: 0.0544 - val_accuracy: 1.0000 - val_loss: 2.9618e-05
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 88s 3s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 9.2112e-05
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 1.0000 - loss: 3.0651e-06 - val_accuracy: 1.0000 - val_loss: 1.3007e-05
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 4.6380e-07
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 1.0012e-07
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 3.4925e-08
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 1.0000 - loss: 2.3283e-10 - val_accuracy: 1.0000 - val_loss: 5.5879e-09
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 1.00

ValueError: Number of classes, 1, does not match size of target_names, 4. Try specifying the labels parameter

In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, Input, Model
from tensorflow.keras.callbacks import EarlyStopping

# -------------------------
# 1. Load CSV
# -------------------------
csv_path = "gas_data/datasets/Gas_Sensors_Measurements.csv"
df = pd.read_csv(csv_path)

sensor_cols = ['MQ2','MQ3','MQ5','MQ6','MQ7','MQ8','MQ135']

# Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Gas'])
class_names = list(label_encoder.classes_)   # proper ordered labels

# -------------------------
# 2. Load images & sensors
# -------------------------
IMG_DIR = "gas_data/datasets/Images"
IMG_SIZE = (128,128)

X_img, X_sensor, y_labels = [], [], []

for _, row in df.iterrows():
    img_path = os.path.join(IMG_DIR, row['Corresponding Image Name'] + ".png")
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue

    img = cv2.resize(img, IMG_SIZE) / 255.0
    X_img.append(img.reshape(128,128,1))
    X_sensor.append(row[sensor_cols].values)
    y_labels.append(row['label'])

X_img = np.array(X_img)
X_sensor = np.array(X_sensor)
y = to_categorical(np.array(y_labels), num_classes=len(class_names))

# Scale sensor data
scaler = MinMaxScaler()
X_sensor = scaler.fit_transform(X_sensor)

# -------------------------
# 3. Train-Test Split (stratified)
# -------------------------
X_img_train, X_img_test, X_sensor_train, X_sensor_test, y_train, y_test = train_test_split(
    X_img, X_sensor, y, test_size=0.2, random_state=42, stratify=y_labels
)

# -------------------------
# 4. Build CNN + Sensor fusion model
# -------------------------
# Image branch
img_input = Input(shape=(128,128,1))
x = layers.Conv2D(32,(3,3),activation='relu',padding='same')(img_input)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(64,(3,3),activation='relu',padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(128,(3,3),activation='relu',padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Flatten()(x)
x = layers.Dense(128,activation='relu')(x)
x = layers.Dropout(0.4)(x)

# Sensor branch
sensor_input = Input(shape=(X_sensor.shape[1],))
s = layers.Dense(64,activation='relu')(sensor_input)
s = layers.BatchNormalization()(s)
s = layers.Dense(32,activation='relu')(s)

# Merge
combined = layers.concatenate([x,s])
z = layers.Dense(64,activation='relu')(combined)
z = layers.Dropout(0.3)(z)
output = layers.Dense(len(class_names),activation='softmax')(z)

model = Model(inputs=[img_input,sensor_input],outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# -------------------------
# 5. Train
# -------------------------
es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    [X_img_train,X_sensor_train], y_train,
    epochs=20, batch_size=32, validation_split=0.2,
    callbacks=[es], verbose=1
)

# -------------------------
# 6. Evaluate
# -------------------------
y_pred_probs = model.predict([X_img_test, X_sensor_test])
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\n✅ Accuracy:", accuracy_score(y_true, y_pred))
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# -------------------------
# 7. Prediction Function
# -------------------------
def predict_new_sample(image_path, sensor_vals):
    # Image preprocessing
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    img = cv2.resize(img, IMG_SIZE) / 255.0
    img = img.reshape(1,128,128,1)

    # Sensor preprocessing
    sensor_arr = np.array(sensor_vals).reshape(1,-1)
    sensor_arr = scaler.transform(sensor_arr)

    # Prediction
    probs = model.predict([img,sensor_arr])[0]
    idx = np.argmax(probs)
    label = class_names[idx]  # proper class from CSV
    confidence = probs[idx]*100
    threat = "Dangerous" if label!="NoGas" else "Safe"

    print("\nPrediction Result")
    print(f"Predicted Gas Type : {label}")
    print(f"Confidence         : {confidence:.2f}%")
    print(f"Threat Level       : {threat}")
    return label, confidence, threat

# -------------------------
# 8. Example usage
# -------------------------
sample_sensor = [560,518,377,339,667,452,418]
sample_image = "gas_data/datasets/Images/999_Mixture.png"
predict_new_sample(sample_image, sample_sensor)

sample_sensor = [560,518,377,339,667,452,418]
sample_image = "gas_data/datasets/Images/1200_Mixture.png"
predict_new_sample(sample_image, sample_sensor)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32768)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │        512 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │  4,194,432 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 160)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │     10,304 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 4)         │        260 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 4,300,772 (16.41 MB)

 Trainable params: 4,300,516 (16.41 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 41s 928ms/step - accuracy: 0.9746 - loss: 0.0437 - val_accuracy: 1.0000 - val_loss: 8.8476e-09
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 28s 874ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 2.2858e-06
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 30s 949ms/step - accuracy: 0.9990 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 3.2596e-09
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 37s 812ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 24s 742ms/step - accuracy: 0.9990 - loss: 0.0212 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 27s 846ms/step - accuracy: 0.9990 - loss: 0.0140 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 29s 901ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 155ms/step

✅ Accuracy: 1.0

ValueError: Number of classes, 1, does not match size of target_names, 4. Try specifying the labels parameter